In [2]:
import pandas as pd

from data_preprocessing import run as preprocess
from feature_engineering import run as featurize
from train_val_split import split_and_encode
from pointwise_fm import fit_encoders, transform, train_fm

preprocess("/local/data/ipv577/expedia-hotel-ranking/data/training_set_VU_DM.csv", "/local/data/ipv577/expedia-hotel-ranking/data/test_set_VU_DM.csv")
featurize("/local/data/ipv577/expedia-hotel-ranking/data/train_clean.parquet", "/local/data/ipv577/expedia-hotel-ranking/data/test_clean.parquet")

# 1. Split + Target-Encodings (Polars)
df_train, df_val = split_and_encode("train_features.parquet")

# 2. Polars → Pandas
df_train_pd = df_train.to_pandas()
df_val_pd   = df_val.to_pandas()

# 3. target-Spalte
df_train_pd["target"] = (
    (df_train_pd["click_bool"] == 1) | (df_train_pd["booking_bool"] == 1)
).astype("float32")

# 4. Downsampling (nur Train)
pos = df_train_pd[df_train_pd["target"] == 1]
neg = df_train_pd[df_train_pd["target"] == 0].sample(n=len(pos)*5, random_state=42)
df_train_bal = pd.concat([pos, neg]).sample(frac=1, random_state=42)

Geladen: 4,958,347 Zeilen, 54 Spalten — Train
Geladen: 4,959,183 Zeilen, 50 Spalten — Test
Preis-Clipping: [14.21, 2061.34]
Gespeichert: train_clean.parquet ((4958347, 52)), test_clean.parquet ((4959183, 50))
Feature Engineering Train...
Feature Engineering Test...
Prop-level Aggregate (train+test kombiniert)...
prop-level Aggregate: 136,886 Hotels, 26 Spalten
Gespeichert: train_features (4958347, 142), test_features (4959183, 139)
Geladen: 4,958,347 Zeilen
Train: 3,966,682 Zeilen, 159,836 Suchen
Val:   991,665 Zeilen,   39,959 Suchen
Target-Encodings hinzugefügt: prop_booking_rate, prop_click_rate, dest_booking_rate, site_booking_rate


In [3]:
# 5. FM
encoders, field_dims = fit_encoders(df_train_bal, n_bins=32)
X_tr, _ = transform(df_train_bal, encoders)
X_va, _ = transform(df_val_pd,   encoders)
model, best_ndcg = train_fm(X_tr, df_train_bal["target"].to_numpy("float32"),
                             X_va, df_val_pd, field_dims)

  Epoche  1 | train loss 0.5087 | Val NDCG@5 0.2847
  Epoche  2 | train loss 0.4246 | Val NDCG@5 0.3052
  Epoche  3 | train loss 0.4183 | Val NDCG@5 0.3166
  Epoche  4 | train loss 0.3906 | Val NDCG@5 0.3225
  Epoche  5 | train loss 0.3971 | Val NDCG@5 0.3323
  Epoche  6 | train loss 0.3928 | Val NDCG@5 0.3335
  Epoche  7 | train loss 0.3774 | Val NDCG@5 0.3365
  Epoche  8 | train loss 0.3672 | Val NDCG@5 0.3368
  Epoche  9 | train loss 0.3755 | Val NDCG@5 0.3440
  Epoche 10 | train loss 0.3789 | Val NDCG@5 0.3450
  Epoche 11 | train loss 0.3863 | Val NDCG@5 0.3444
  Epoche 12 | train loss 0.3591 | Val NDCG@5 0.3484
  Epoche 13 | train loss 0.3720 | Val NDCG@5 0.3494
  Epoche 14 | train loss 0.3824 | Val NDCG@5 0.3512
  Epoche 15 | train loss 0.3682 | Val NDCG@5 0.3495
  Epoche 16 | train loss 0.3443 | Val NDCG@5 0.3523
  Epoche 17 | train loss 0.3567 | Val NDCG@5 0.3513
  Epoche 18 | train loss 0.3597 | Val NDCG@5 0.3521
  Epoche 19 | train loss 0.3482 | Val NDCG@5 0.3519
  Early Stop

In [5]:
import importlib
import pointwise_fm  # oder welches Modul auch immer
importlib.reload(pointwise_fm)

# danach neu importieren
from pointwise_fm import fit_encoders, transform, train_fm

# 5. FM; changed emb_dim 16 > 32; patience 3 > 7, epochs 20 > 40
encoders, field_dims = fit_encoders(df_train_bal, n_bins=32)
X_tr, _ = transform(df_train_bal, encoders)
X_va, _ = transform(df_val_pd,   encoders)
model, best_ndcg = train_fm(X_tr, df_train_bal["target"].to_numpy("float32"),
                             X_va, df_val_pd, field_dims)

  Epoche  1 | train loss 0.4708 | Val NDCG@5 0.2981
  Epoche  2 | train loss 0.4196 | Val NDCG@5 0.3148
  Epoche  3 | train loss 0.4013 | Val NDCG@5 0.3264
  Epoche  4 | train loss 0.3998 | Val NDCG@5 0.3305
  Epoche  5 | train loss 0.3925 | Val NDCG@5 0.3387
  Epoche  6 | train loss 0.3517 | Val NDCG@5 0.3389
  Epoche  7 | train loss 0.3742 | Val NDCG@5 0.3380
  Epoche  8 | train loss 0.3753 | Val NDCG@5 0.3397
  Epoche  9 | train loss 0.3571 | Val NDCG@5 0.3434
  Epoche 10 | train loss 0.3588 | Val NDCG@5 0.3454
  Epoche 11 | train loss 0.3580 | Val NDCG@5 0.3456
  Epoche 12 | train loss 0.3494 | Val NDCG@5 0.3450
  Epoche 13 | train loss 0.3681 | Val NDCG@5 0.3476
  Epoche 14 | train loss 0.3715 | Val NDCG@5 0.3449
  Epoche 15 | train loss 0.3515 | Val NDCG@5 0.3441
  Epoche 16 | train loss 0.3711 | Val NDCG@5 0.3461
  Epoche 17 | train loss 0.3661 | Val NDCG@5 0.3450
  Epoche 18 | train loss 0.3380 | Val NDCG@5 0.3504
  Epoche 19 | train loss 0.3395 | Val NDCG@5 0.3493
  Epoche 20 

## pairwise_fm

In [1]:
from pointwise_fm import fit_encoders, transform
from train_val_split import split_and_encode
import numpy as np
import torch
import importlib

import pairwise_fm 
importlib.reload(pairwise_fm)

# danach neu importieren
from pairwise_fm import build_pair_index, train_pairwise_fm

# 1. Split + Target-Encodings (Polars macht das rasend schnell)
df_train_pl, df_val_pl = split_and_encode("train_features.parquet")

# 2. Polars → Pandas (Wir überschreiben die Variablen direkt, 
# um Fehler im restlichen Code zu vermeiden)
df_train = df_train_pl.to_pandas()
df_val   = df_val_pl.to_pandas()

# 3. WICHTIG: Das Relevance-Label muss da sein (0=Ignoriert, 1=Klick, 5=Buchung)
for df in [df_train, df_val]:
    if 'relevance' not in df.columns:
         df['relevance'] = np.where(df["booking_bool"] == 1, 5,
                           np.where(df["click_bool"] == 1, 1, 0))

# 4. Features in Integer/Kategorien umwandeln (Encoder fitten & anwenden)
# Hier greifen wir jetzt sauber auf die Pandas-DataFrames zu
encoders, field_dims = fit_encoders(df_train, n_bins=32)
X_tr, _ = transform(df_train, encoders)
X_va, _ = transform(df_val, encoders)

# 5. PAAR-BILDUNG (Der neue Schritt für Pairwise!)
# Sucht in df_train nach Gewinner-Verlierer-Paaren pro srch_id
pair_idx = build_pair_index(df_train, max_pairs_per_group=30)

# 6. Training starten!
model, best_ndcg = train_pairwise_fm(
    X_tr=X_tr, 
    pair_idx=pair_idx, 
    X_va=X_va, 
    df_val=df_val, 
    field_dims=field_dims,
    epochs=20 # 10 ist super für den ersten Test-Lauf!
)

# evaluation of computed scores per hotel; does the model collapse and predicts zero for every hotel, even when it is a booked one?
Xva_t = torch.from_numpy(np.array(X_va, copy=True)).to("cuda" if torch.cuda.is_available() else "cpu")

model.eval()
with torch.no_grad():
    scores = model(Xva_t).cpu().numpy()

print("Score-Verteilung:")
print(f"  Min:    {scores.min():.3f}")
print(f"  Max:    {scores.max():.3f}")
print(f"  Mean:   {scores.mean():.3f}")
print(f"  Std:    {scores.std():.3f}")

# Wichtig: schauen ob Scores sich zwischen relevanten und irrelevanten Hotels unterscheiden
df_val_pd["score"] = scores
print("\nScore nach Relevanz:")
print(df_val_pd.groupby("relevance")["score"].describe())

Geladen: 4,958,347 Zeilen
Train: 3,966,682 Zeilen, 159,836 Suchen
Val:   991,665 Zeilen,   39,959 Suchen
Target-Encodings hinzugefügt: prop_booking_rate, prop_click_rate, dest_booking_rate, site_booking_rate
Erstelle Paar-Indizes (Pair Sampling)...
-> 3,751,244 Trainings-Paare generiert.
Starte Training auf cuda...
  Epoche  1 | BPR Loss: 0.4957 | Val NDCG@5: 0.3677
  Epoche  2 | BPR Loss: 0.4126 | Val NDCG@5: 0.3767
  Epoche  3 | BPR Loss: 0.3963 | Val NDCG@5: 0.3800
  Epoche  4 | BPR Loss: 0.3846 | Val NDCG@5: 0.3790
  Epoche  5 | BPR Loss: 0.3772 | Val NDCG@5: 0.3793
  Epoche  6 | BPR Loss: 0.3732 | Val NDCG@5: 0.3800
  Epoche  7 | BPR Loss: 0.3712 | Val NDCG@5: 0.3792
  Epoche  8 | BPR Loss: 0.3702 | Val NDCG@5: 0.3771
  Epoche  9 | BPR Loss: 0.3698 | Val NDCG@5: 0.3782
  Epoche 10 | BPR Loss: 0.3695 | Val NDCG@5: 0.3773
  Epoche 11 | BPR Loss: 0.3692 | Val NDCG@5: 0.3767
  Epoche 12 | BPR Loss: 0.3690 | Val NDCG@5: 0.3756
  Epoche 13 | BPR Loss: 0.3689 | Val NDCG@5: 0.3792
  Early

NameError: name 'df_val_pd' is not defined

In [5]:
pairs = build_pair_index(df_train_pd, max_pairs_per_group=30)

# Stichprobe: schauen ob pos wirklich relevanter ist als neg
sample = pairs[np.random.choice(len(pairs), 1000, replace=False)]
rel = df_train_pd["relevance"].to_numpy()
pos_rel = rel[sample[:, 0]]
neg_rel = rel[sample[:, 1]]

print(f"Pos relevance: mean={pos_rel.mean():.2f}, min={pos_rel.min()}, max={pos_rel.max()}")
print(f"Neg relevance: mean={neg_rel.mean():.2f}, min={neg_rel.min()}, max={neg_rel.max()}")
assert (pos_rel > neg_rel).mean() > 0.99, "Paar-Reihenfolge stimmt nicht!"
print("Paar-Qualität: OK")

Erstelle Paar-Indizes (Pair Sampling)...
-> 3,751,244 Trainings-Paare generiert.
Pos relevance: mean=3.72, min=1, max=5
Neg relevance: mean=0.00, min=0, max=0
Paar-Qualität: OK


In [ ]:
from pointwise_fm import fit_encoders, transform
from pairwise_fm_v2 import build_pairs, train_pairwise_fm, train_lambda_fm

# 1. Encoders fitten und transformieren (df_train_pd = volles Train, kein Downsampling)
encoders, field_dims = fit_encoders(df_train_pd, n_bins=32)
X_tr, _ = transform(df_train_pd, encoders)
X_va, _ = transform(df_val_pd,   encoders)

# 2. Gestufte Paare bauen
pairs = build_pairs(df_train_pd, max_pairs_per_group=50)

# 3a. Variante A — nur gestufte Paare (schneller, guter erster Test)
model, best_ndcg = train_pairwise_fm(
    X_tr       = X_tr,
    pairs      = pairs,
    X_va       = X_va,
    df_val     = df_val_pd,
    field_dims = field_dims,
    embed_dim  = 32,
    l2         = 1e-3,
    lr         = 1e-3,
    patience   = 5,
    epochs     = 30,
)

# 3b. Variante B — gestufte Paare + ΔNDCG (LambdaFM, stärker)
rel_tr = df_train_pd["relevance"].to_numpy()
grp_tr = df_train_pd["srch_id"].to_numpy()

model, best_ndcg = train_lambda_fm(
    X_tr       = X_tr,
    pairs      = pairs,
    rel_tr     = rel_tr,
    grp_tr     = grp_tr,
    X_va       = X_va,
    df_val     = df_val_pd,
    field_dims = field_dims,
    embed_dim  = 32,
    l2         = 1e-3,
    lr         = 1e-3,
    patience   = 5,
    epochs     = 30,
)

# 4. Score-Check nach dem Training
import torch
import numpy as np

device  = "cuda" if torch.cuda.is_available() else "cpu"
Xva_cpu = torch.from_numpy(np.array(X_va, copy=True))

model.eval()
scores_list = []
with torch.no_grad():
    for i in range(0, len(Xva_cpu), 8192):
        batch = Xva_cpu[i:i + 8192].to(device)
        scores_list.append(model(batch).cpu())
scores = torch.cat(scores_list).numpy()

import pandas as pd
df_check = pd.DataFrame({"relevance": df_val_pd["relevance"].to_numpy(), "score": scores})
print(df_check.groupby("relevance")["score"].describe()[["mean", "std", "min", "max"]])

In [2]:
from pointwise_fm import fit_encoders, transform
from train_val_split import split_and_encode
import numpy as np
import torch
import importlib

import pairwise_fm 
importlib.reload(pairwise_fm)

# danach neu importieren
from pairwise_fm import build_pair_index, train_pairwise_fm

from pointwise_fm import fit_encoders, transform
from pairwise_fm_v2 import build_pairs
from deep_fm import DeepFMScorer, train_deep_lambda_fm, run_optuna

df_train_pl, df_val_pl = split_and_encode("train_features.parquet")

# 2. Polars → Pandas (Wir überschreiben die Variablen direkt, 
# um Fehler im restlichen Code zu vermeiden)
df_train_pd = df_train_pl.to_pandas()
df_val_pd   = df_val_pl.to_pandas()

encoders, field_dims = fit_encoders(df_train_pd, n_bins=32)
X_tr, _ = transform(df_train_pd, encoders)
X_va, _ = transform(df_val_pd,   encoders)

pairs    = build_pairs(df_train_pd, max_pairs_per_group=50)
rel_tr   = df_train_pd["relevance"].to_numpy()
grp_tr   = df_train_pd["srch_id"].to_numpy()


/local/data/ipv577/expedia-hotel-ranking/.dm_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Geladen: 4,958,347 Zeilen
Train: 3,966,682 Zeilen, 159,836 Suchen
Val:   991,665 Zeilen,   39,959 Suchen
Target-Encodings hinzugefügt: prop_booking_rate, prop_click_rate, dest_booking_rate, site_booking_rate
Erstelle gestufte Paar-Indizes...
  rel=5 vs rel=0 (w=1.0): 2,652,581 Paare
  rel=5 vs rel=1 (w=0.7): 11,187 Paare
  rel=1 vs rel=0 (w=0.4): 1,410,707 Paare
  Gesamt: 4,074,475 Paare


In [5]:
model, best_ndcg = train_deep_lambda_fm(
    X_tr, pairs, rel_tr, grp_tr, X_va, df_val_pd, field_dims,
    embed_dim  = 32,
    mlp_dims   = (256, 128, 64),
    dropout    = 0.2,
    lr         = 1e-3,
    l2         = 1e-3,
    epochs     = 30,
    patience   = 5,
)

  Epoche  1 | Loss 0.0162 | Val NDCG@5 0.3383 | mean Δw 0.1344
  Epoche  2 | Loss 0.0024 | Val NDCG@5 0.3488 | mean Δw 0.2437
  Epoche  3 | Loss 0.0023 | Val NDCG@5 0.3355 | mean Δw 0.2497
  Epoche  4 | Loss 0.0024 | Val NDCG@5 0.3538 | mean Δw 0.2434
  Epoche  5 | Loss 0.0022 | Val NDCG@5 0.3508 | mean Δw 0.2531
  Epoche  6 | Loss 0.0021 | Val NDCG@5 0.3588 | mean Δw 0.2524
  Epoche  7 | Loss 0.0022 | Val NDCG@5 0.3304 | mean Δw 0.2548
  Epoche  8 | Loss 0.0021 | Val NDCG@5 0.3576 | mean Δw 0.2300
  Epoche  9 | Loss 0.0021 | Val NDCG@5 0.3589 | mean Δw 0.2550
  Epoche 10 | Loss 0.0021 | Val NDCG@5 0.3657 | mean Δw 0.2560
  Epoche 11 | Loss 0.0021 | Val NDCG@5 0.3608 | mean Δw 0.2584
  Epoche 12 | Loss 0.0021 | Val NDCG@5 0.3615 | mean Δw 0.2572


KeyboardInterrupt: 

In [3]:
model, best_ndcg = train_deep_lambda_fm(
    X_tr, pairs, rel_tr, grp_tr, X_va, df_val_pd, field_dims,
    embed_dim  = 16,              # war 32 — kleiner
    mlp_dims   = (128, 64),       # war (256,128,64) — kleiner
    dropout    = 0.4,             # war 0.2 — aggressiver
    lr         = 3e-4,            # war 1e-3 — langsamer
    l2         = 5e-3,            # war 1e-3 — stärker
    epochs     = 30,
    patience   = 5,
)

  Epoche  1 | Loss 0.1125 | Val NDCG@5 0.3534 | mean Δw 0.1264
  Epoche  2 | Loss 0.0720 | Val NDCG@5 0.3641 | mean Δw 0.2540
  Epoche  3 | Loss 0.0700 | Val NDCG@5 0.3644 | mean Δw 0.2594
  Epoche  4 | Loss 0.0693 | Val NDCG@5 0.3666 | mean Δw 0.2601
  Epoche  5 | Loss 0.0688 | Val NDCG@5 0.3679 | mean Δw 0.2615
  Epoche  6 | Loss 0.0687 | Val NDCG@5 0.3701 | mean Δw 0.2626
  Epoche  7 | Loss 0.0686 | Val NDCG@5 0.3699 | mean Δw 0.2631
  Epoche  8 | Loss 0.0686 | Val NDCG@5 0.3708 | mean Δw 0.2640
  Epoche  9 | Loss 0.0686 | Val NDCG@5 0.3718 | mean Δw 0.2643
  Epoche 10 | Loss 0.0687 | Val NDCG@5 0.3716 | mean Δw 0.2649
  Epoche 11 | Loss 0.0686 | Val NDCG@5 0.3711 | mean Δw 0.2648
  Epoche 12 | Loss 0.0687 | Val NDCG@5 0.3715 | mean Δw 0.2651
  Epoche 13 | Loss 0.0687 | Val NDCG@5 0.3718 | mean Δw 0.2649
  Epoche 14 | Loss 0.0687 | Val NDCG@5 0.3720 | mean Δw 0.2650
  Epoche 15 | Loss 0.0687 | Val NDCG@5 0.3716 | mean Δw 0.2649
  Epoche 16 | Loss 0.0687 | Val NDCG@5 0.3716 | mean Δw

KeyboardInterrupt: 

In [ ]:
# optuna search

best_params, study = run_optuna(
    X_tr, pairs, rel_tr, grp_tr, X_va, df_val_pd, field_dims,
    n_trials          = 20,   # wie viele Kombinationen ausprobieren
    epochs_per_trial  = 8,    # kurz genug um schnell zu sein
    patience_per_trial= 3,
)

# Danach finaler Run mit besten Parametern:
from pairwise_fm_v2 import build_pairs
mlp_map = {
    "small":  (128, 64),
    "medium": (256, 128, 64),
    "large":  (512, 256, 128, 64),
    "deep":   (256, 256, 128, 64),
}
model_final, ndcg_final = train_deep_lambda_fm(
    X_tr, pairs, rel_tr, grp_tr, X_va, df_val_pd, field_dims,
    embed_dim = best_params["embed_dim"],
    mlp_dims  = mlp_map[best_params["mlp_arch"]],
    dropout   = best_params["dropout"],
    lr        = best_params["lr"],
    l2        = best_params["l2"],
    epochs    = 40,
    patience  = 6,
)